# AeroConform Evaluation

Full evaluation protocol: detection rates per attack type, calibration plots, ADE/FDE.

In [ ]:
# Cell 1: Setup
import subprocess, sys, os
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'torch-geometric', 'polars', 'pyarrow',
    'structlog', 'pyyaml', 'scipy', 'matplotlib', 'tqdm', 'rich'])

if os.path.exists('/content'):
    if not os.path.exists('aeroconform'):
        subprocess.check_call(['git', 'clone', 'https://github.com/Nadosaurusrex/aeroconform.git'])
    os.chdir('aeroconform')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])
    sys.path.insert(0, '.')

from pathlib import Path
if os.path.exists('/content'):
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/aeroconform/data')
    CHECKPOINT_DIR = Path('/content/drive/MyDrive/aeroconform/checkpoints')
else:
    DATA_DIR = Path('data')
    CHECKPOINT_DIR = Path('checkpoints')

print('Setup complete')

In [ ]:
# Cell 2: Load model and data
import torch
import numpy as np
import polars as pl
from src.models.aerogpt import AeroGPT
from src.models.conformal import AdaptiveConformal
from src.data.flight_segmentation import segment_flights
from src.data.preprocessing import delta_encode, normalize, compute_norm_stats
from src.data.schemas import NormStats
from src.data.synthetic import (
    inject_position_spoofing, inject_ghost,
    inject_gps_drift, inject_impossible_maneuver
)
from src.evaluation.metrics import compute_detection_rate, compute_calibration_error
from src.utils.config import ModelConfig
from src.utils.logging import setup_logging

setup_logging(level='INFO')

# Load model
model_config = ModelConfig.from_yaml(Path('configs/model.yaml'))
model = AeroGPT(model_config)
ckpt = torch.load(CHECKPOINT_DIR / 'aerogpt_final.pt', map_location='cpu', weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Model loaded: {model.count_parameters():,} parameters')

# Load data
norm_stats = NormStats.load(str(DATA_DIR / 'norm_stats.npz'))
parquet_files = sorted((DATA_DIR / 'raw').glob('*.parquet'))

# Use last 10% of files as test set
test_files = parquet_files[-max(1, len(parquet_files)//10):]
test_flights = []
for pf in test_files:
    df = pl.read_parquet(pf)
    flights = segment_flights(df)
    test_flights.extend(flights)

print(f'Test flights: {len(test_flights)}')

In [ ]:
# Cell 3: Calibrate conformal detector
conformal = AdaptiveConformal(alpha=0.01, buffer_size=2000, decay=0.995)

# Use first 50 test flights to calibrate
cal_flights = test_flights[:50]
cal_scores = []

with torch.no_grad():
    for flight in cal_flights:
        deltas = delta_encode(flight.features)
        normalized = normalize(deltas, norm_stats)
        elapsed = np.cumsum(np.diff(flight.timestamps, prepend=flight.timestamps[0])).astype(np.float32)
        
        x = torch.from_numpy(normalized[:-1]).unsqueeze(0)
        t = torch.from_numpy(elapsed[:-1]).unsqueeze(0)
        means, log_vars, _ = model(x, t)
        
        for step in range(means.shape[1]):
            from src.models.conformal import mahalanobis_score
            s = mahalanobis_score(
                normalized[step + 1],
                means[0, step].numpy(),
                log_vars[0, step].numpy()
            )
            cal_scores.append(s)

conformal.calibrate(cal_scores)
print(f'Calibrated with {len(cal_scores)} scores')

In [ ]:
# Cell 4: Evaluate on synthetic anomalies
results = {}
eval_flights = test_flights[50:70]  # Use 20 flights for anomaly injection

for anomaly_type, inject_fn in [
    ('spoofing', inject_position_spoofing),
    ('gps_drift', inject_gps_drift),
    ('impossible_maneuver', inject_impossible_maneuver),
]:
    all_preds = []
    all_labels = []
    
    for flight in eval_flights:
        lf = inject_fn(flight)
        labels = np.zeros(flight.num_steps, dtype=np.bool_)
        for label in lf.labels:
            labels[label.start_idx:label.end_idx] = True
        
        deltas = delta_encode(lf.flight.features)
        normalized = normalize(deltas, norm_stats)
        elapsed = np.arange(lf.flight.num_steps, dtype=np.float32) * 10
        
        preds = np.zeros(lf.flight.num_steps, dtype=np.bool_)
        with torch.no_grad():
            x = torch.from_numpy(normalized[:-1]).unsqueeze(0)
            t = torch.from_numpy(elapsed[:-1]).unsqueeze(0)
            means, log_vars, _ = model(x, t)
            
            for step in range(means.shape[1]):
                result = conformal.score(
                    normalized[step + 1],
                    means[0, step].numpy(),
                    log_vars[0, step].numpy()
                )
                preds[step + 1] = result.is_anomaly
        
        all_preds.append(preds)
        all_labels.append(labels)
    
    preds_cat = np.concatenate(all_preds)
    labels_cat = np.concatenate(all_labels)
    result = compute_detection_rate(preds_cat, labels_cat)
    results[anomaly_type] = result
    print(f'{anomaly_type}: recall={result.detection_rate:.3f}, FAR={result.false_alarm_rate:.4f}')

# Ghost injection
all_preds = []
all_labels = []
for i in range(20):
    lf = inject_ghost(seed=i)
    labels = np.ones(lf.flight.num_steps, dtype=np.bool_)
    deltas = delta_encode(lf.flight.features)
    normalized = normalize(deltas, norm_stats)
    elapsed = np.arange(lf.flight.num_steps, dtype=np.float32) * 10
    
    preds = np.zeros(lf.flight.num_steps, dtype=np.bool_)
    with torch.no_grad():
        x = torch.from_numpy(normalized[:-1]).unsqueeze(0)
        t = torch.from_numpy(elapsed[:-1]).unsqueeze(0)
        means, log_vars, _ = model(x, t)
        for step in range(means.shape[1]):
            result = conformal.score(
                normalized[step + 1],
                means[0, step].numpy(),
                log_vars[0, step].numpy()
            )
            preds[step + 1] = result.is_anomaly
    
    all_preds.append(preds)
    all_labels.append(labels)

preds_cat = np.concatenate(all_preds)
labels_cat = np.concatenate(all_labels)
ghost_result = compute_detection_rate(preds_cat, labels_cat)
results['ghost'] = ghost_result
print(f'ghost: recall={ghost_result.detection_rate:.3f}, FAR={ghost_result.false_alarm_rate:.4f}')

In [ ]:
# Cell 5: Summary and plots
import matplotlib.pyplot as plt

print('\n=== EVALUATION RESULTS ===')
print(f'Target alpha: 0.01')
for name, result in results.items():
    status = 'PASS' if (
        (name in ['spoofing', 'impossible_maneuver'] and result.detection_rate > 0.95) or
        (name in ['ghost', 'gps_drift'] and result.detection_rate > 0.80)
    ) else 'FAIL'
    print(f'  {name}: recall={result.detection_rate:.3f}, FAR={result.false_alarm_rate:.4f} [{status}]')

# Bar chart of detection rates
fig, ax = plt.subplots(figsize=(8, 5))
names = list(results.keys())
recalls = [results[n].detection_rate for n in names]
thresholds = [0.95, 0.80, 0.80, 0.95]  # spoofing, ghost, drift, maneuver

bars = ax.bar(names, recalls, color=['#22c55e' if r > t else '#ef4444' for r, t in zip(recalls, thresholds)])
for i, t in enumerate(thresholds):
    ax.axhline(y=t, xmin=i/len(names), xmax=(i+1)/len(names), color='gray', linestyle='--', alpha=0.5)

ax.set_ylabel('Detection Rate (Recall)')
ax.set_title('AeroConform Anomaly Detection Results')
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(str(CHECKPOINT_DIR / 'detection_rates.png'), dpi=150)
plt.show()